# Colab Evaluation Workflow

Run baseline and LoRA generation, then score TIFA and GenAI-Bench with API judges.

## 1. Environment Setup

In [4]:
%cd /content/T2I-RL-Eval
!pip install -r requirements.txt

/content/T2I-RL-Eval
  Cloning https://github.com/deepseek-ai/Janus.git to /tmp/pip-install-r40kvpgl/janus_d9faa6f22c5747b3b833ff684ea8d82a
  Running command git clone --filter=blob:none --quiet https://github.com/deepseek-ai/Janus.git /tmp/pip-install-r40kvpgl/janus_d9faa6f22c5747b3b833ff684ea8d82a
  Resolved https://github.com/deepseek-ai/Janus.git to commit 1daa72fa409002d40931bd7b36a9280362469ead
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 130.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.

## 2. Configuration

In [10]:
import os
API_KEY = 'YOUR_API_KEY'
API_BASE_URL = 'https://api.openai.com/v1'
JUDGE_MODEL = 'gpt-4.1-mini'
JUDGE_MAX_WORKERS = 4
JUDGE_LOG_EVERY = 20

os.environ['OPENAI_API_KEY'] = API_KEY
os.environ['OPENAI_BASE_URL'] = API_BASE_URL
os.environ['JUDGE_MODEL'] = JUDGE_MODEL

BASE_MODEL = 'deepseek-ai/Janus-Pro-1B'
LORA_DIR = 'artifacts/lora/grpo_siliconflow_quick_final'
OUTPUT_DIR = 'outputs/evaluation'

import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/T2I-RL-Eval")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


## 3. Generate Before Images

In [12]:
%cd /content/T2I-RL-Eval
!python scripts/generate_benchmark_images.py \
        --benchmark tifa --variant before \
        --manifest_path data/evaluation/tifa/samples.jsonl \
        --base_model $BASE_MODEL \
        --output_dir $OUTPUT_DIR \
        --resume \
        --dtype float16 \
        --prompt_batch_size 4

/content/T2I-RL-Eval
2026-04-09 11:07:39.995018: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-09 11:07:40.509857: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775732860.691552    3500 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775732860.740311    3500 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775732861.166443    3500 computation_placer.cc:177] computation placer already registered. Please check linka

In [ ]:
%cd /content/T2I-RL-Eval
!python scripts/generate_benchmark_images.py \
        --benchmark genai_bench --variant before \
        --manifest_path data/evaluation/genai_bench/samples.jsonl \
        --base_model $BASE_MODEL \
        --output_dir $OUTPUT_DIR \
        --resume \
        --dtype float16 \
        --prompt_batch_size 4

## 4. Generate After Images

In [ ]:
!python scripts/generate_benchmark_images.py \
        --benchmark tifa --variant after \
        --manifest_path data/evaluation/tifa/samples.jsonl \
        --base_model $BASE_MODEL \
        --lora_path $LORA_DIR \
        --output_dir $OUTPUT_DIR \
        --resume \
        --dtype float16 \
        --prompt_batch_size 12
!python scripts/generate_benchmark_images.py \
        --benchmark genai_bench --variant after \
        --manifest_path data/evaluation/genai_bench/samples.jsonl \
        --base_model $BASE_MODEL \
        --lora_path $LORA_DIR \
        --output_dir $OUTPUT_DIR \
        --resume \
        --dtype float16 \
        --prompt_batch_size 12


2026-04-09 12:18:35.634394: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-09 12:18:35.652754: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775737115.675098   17504 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775737115.681800   17504 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775737115.699952   17504 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## 5. Run TIFA

In [ ]:
!python scripts/run_tifa.py --manifest_path data/evaluation/tifa/samples.jsonl --images_root outputs/evaluation/images --variant before --output_path outputs/evaluation/results/tifa_before.jsonl --resume --max_workers $JUDGE_MAX_WORKERS --log_every $JUDGE_LOG_EVERY
!python scripts/run_tifa.py --manifest_path data/evaluation/tifa/samples.jsonl --images_root outputs/evaluation/images --variant after --output_path outputs/evaluation/results/tifa_after.jsonl --resume --max_workers $JUDGE_MAX_WORKERS --log_every $JUDGE_LOG_EVERY


## 6. Run GenAI-Bench

In [ ]:
!python scripts/run_genai_bench.py --manifest_path data/evaluation/genai_bench/samples.jsonl --images_root outputs/evaluation/images --variant before --output_path outputs/evaluation/results/genai_before.jsonl --resume --max_workers $JUDGE_MAX_WORKERS --log_every $JUDGE_LOG_EVERY
!python scripts/run_genai_bench.py --manifest_path data/evaluation/genai_bench/samples.jsonl --images_root outputs/evaluation/images --variant after --output_path outputs/evaluation/results/genai_after.jsonl --resume --max_workers $JUDGE_MAX_WORKERS --log_every $JUDGE_LOG_EVERY


## 7. Summary

In [ ]:
!python scripts/summarize_evaluation.py --tifa_results_before outputs/evaluation/results/tifa_before.jsonl --tifa_results_after outputs/evaluation/results/tifa_after.jsonl --genai_results_before outputs/evaluation/results/genai_before.jsonl --genai_results_after outputs/evaluation/results/genai_after.jsonl --output_dir outputs/evaluation/reports
